# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r boundary_gdf ndvi_da

You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries

import panel as pn

import cartopy.crs as ccrs

### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

## Step 5: Is the mean annual July NDVI different within the Gila River Indian Community after the change in water rights?

You will compute the mean NDVI inside and outside the Reservation boundary. 

### Step 5A: Get the NDVI inside and outside the Gila River Indian Community

First, use the code below to get a `GeoDataFrame` of the area outside the Reservation.

- Check the variable names. Make sure that the code uses your boundary `GeoDataFrame`  
- How could you test if the geometry was modified correctly? Add some code to take a look at the results

In [3]:
### compute the area outside the reservation boundary
out_gdf = (

    ### make the smallest box containing GRIC
    gpd.GeoDataFrame(geometry = boundary_gdf.envelope)

    ### substract the GRIC polygon from the box
    .overlay(boundary_gdf, how = 'difference'))

In [4]:
### figure out the appropriate placement of this chunk
### figure out what the code does and annotate it
if not hasattr(ccrs.PlateCarree, '_original_proj4_params'):
    ccrs.PlateCarree._original_proj4_params = ccrs.PlateCarree.proj4_params

def _proj4_params_with_lon_0(self):
    params = ccrs.PlateCarree._original_proj4_params.fget(self)
    params.setdefault('lon_0', params.get('pm', 0))
    return params

ccrs.PlateCarree.proj4_params = property(_proj4_params_with_lon_0)

In [5]:
### plot the gdf of bounding box outside GRIC
out_gdf.hvplot(

    ### set color
    fill_color = 'red',

    ### set basemap
    geo = True,

    ### set title
    title = 'The Region Outside GRIC'
)

:Polygons   [Longitude,Latitude]

Next, clip your DataArray to the boundaries for both inside and outside
the reservation. You will need to replace the `GeoDataFrame` name with
your own. Check out the [lesson on clipping data with the `rioxarray`
library on EarthDataScience.org](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/).

**Note**: It’s important to use `from_disk = True` when clipping large arrays like this. It allows the computer to use less valuable memory resources when clipping. Otherwise, you will probably find that the cell below crashes your kernel.

In [6]:
### clip data to inside the reservation's boundaries
ndvi_in_da = ndvi_da.rio.clip(boundary_gdf.geometry, from_disk = True)

In [7]:
### clip data to outside the reservation's boundaries 
ndvi_out_da = ndvi_da.rio.clip(out_gdf.geometry, from_disk = True)

In [8]:
ndvi_out_da = ndvi_out_da.assign_coords(year = ndvi_out_da.date.dt.year.values)

ndvi_out_da

<xarray.Dataset> Size: 48MB
Dimensions:      (x: 380, y: 203, date: 154, year: 154)
Coordinates:
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
  * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
  * date         (date) datetime64[us] 1kB 2001-01-14 2001-01-16 ... 2022-01-24
  * year         (year) int64 1kB 2001 2001 2001 2001 ... 2022 2022 2022 2022
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 48MB 0.6146 0.3796 ... 0.1736 0.2146

### Step 5B: Assess the change in NDVI before and after the change in water rights, inside and outside the boundaries of the Gila River Indian Community

For **both inside and outside** the Gila River Indian Community boundary, calculate the mean annual July NDVI:  

- Group the data by year  
- Take the mean. You always need to tell reducing methods in `xarray` what dimensions you want to reduce. When you want to summarize data across **all** dimensions, you can use the `...` syntax, e.g. `.mean(...)` as a shorthand  
- Select the NDVI variable  
- Convert to a DataFrame using the `to_dataframe()` method  
- Join the two DataFrames for plotting using the `.join()` method. You will need to rename the columns using the `lsuffix =` and `rsuffix =` parameters
- Finally, plot annual July means for both inside and outside the Reservation on the same plot.

**Note**: The DateIndex in pandas is a little different from the Datetime Dimension in xarray. You will need to use the `.dt.year` syntax to access information about the year, not just `.year`.

In [9]:
### calculate mean annual July NDVI inside
july_ndvi_in_df = (
    
    ndvi_in_da

    ### group by year
    .groupby(ndvi_in_da.date.dt.year)

    ### take mean of all values in each group
    .mean(...)

    ### extract NDVI and put it in a dataframe
    .NDVI.to_dataframe()  
)

july_ndvi_in_df.head()

,band,spatial_ref,NDVI
year,,,
2001,1,0,0.199645
2002,1,0,0.177933
2003,1,0,0.187301
2004,1,0,0.176162
2005,1,0,0.238630


In [10]:
### calculate mean annual July NDVI outside
july_ndvi_out_df = (
    
    ndvi_out_da

    ### group by year
    .groupby(ndvi_in_da.date.dt.year)

    ### take mean of all values in each group
    .mean(...)

    ### extract NDVI and put it in a dataframe
    .NDVI.to_dataframe()  
)

july_ndvi_out_df.head()

,band,spatial_ref,NDVI
year,,,
2001,1,0,0.247629
2002,1,0,0.226726
2003,1,0,0.229889
2004,1,0,0.221753
2005,1,0,0.255275


In [11]:
### join the two dataframes
july_ndvi_df = (

    ### take the NDVI from inside
    july_ndvi_in_df[['NDVI']]

    ### join to the df from otside
    .join(

        ### take the NDVI from outside
        july_ndvi_out_df[['NDVI']],
        
        ### distinguish NDVI column names
        lsuffix = ' inside GRIC', rsuffix = ' outside GRIC')
)

july_ndvi_df.head()

,NDVI inside GRIC,NDVI outside GRIC
year,,
2001,0.199645,0.247629
2002,0.177933,0.226726
2003,0.187301,0.229889
2004,0.176162,0.221753
2005,0.238630,0.255275


In [12]:
### plot inside and outside the reservation on a single plot
july_ndvi_df.hvplot(

    ### set x coordinate
    x = 'year',

    ### set inside and outside plots
    y = ['NDVI inside GRIC', 'NDVI outside GRIC'],

    ### axis labels
    xlabel = 'Year', ylabel = 'Mean NDVI',

    ### set plot type
    kind = 'line',

    ### set frame width
    frame_width = 600,

    ### set title
    title = 'Yearly Mean NDVI Inside vs. Outside GRIC',

    ### set up legend
    legend = True,
    legend_opts = {
        'title': '',
        'label_text_font_size': '10pt',
        'location': (5, 5)
        }
)

:NdOverlay   [Variable]
   :Curve   [year]   (value)

Now, take the difference between outside and inside the site boundary and plot that


In [13]:
### create a column for difference in July NDVI
july_ndvi_df['NDVI difference'] = (
    july_ndvi_df['NDVI inside GRIC']
    - july_ndvi_df['NDVI outside GRIC']
)

july_ndvi_df

,NDVI inside GRIC,NDVI outside GRIC,NDVI difference
year,,,
2001,0.199645,0.247629,-0.047984
2002,0.177933,0.226726,-0.048793
2003,0.187301,0.229889,-0.042588
2004,0.176162,0.221753,-0.045591
2005,0.238630,0.255275,-0.016645
2006,0.211491,0.235571,-0.024079
2007,0.181710,0.211984,-0.030274
2008,0.201902,0.237739,-0.035836
2009,0.179118,0.218907,-0.039790


In [ ]:
### plot difference inside and outside the boundary
july_ndvi_df['NDVI difference'].hvplot(

    ### set title
    title = 'Yearly Difference in Mean NDVI between Inside and Outside GRIC',

    ### set axis labels
    xlabel = 'Year', ylabel = 'Difference in Mean NDVI',

    ### set plot type
    kind = 'line',

    ### set frame width
    frame_width = 600,
)

:Curve   [year]   (NDVI difference)

In [15]:
### plot yearly mean NDVI inside/outside GRIC

### plot yearly mean ndvi
ndvi_yr_plot = (
    ndvi_da

    ### group by year
    .groupby(ndvi_da.date.dt.year)

    ### take mean of year for each pixel
    .mean('date')

    ### extract NDVI values
    .NDVI

    ### plot
    .hvplot(

        ### set lang/lat
        x = 'x', y = 'y',

        ### set color
        cmap = 'PiYG',

        ### set basemap
        geo = True,

        ### set title
        title = 'Yearly Mean NDVI Inside and Outside GRIC'
    )
)

### plot GRIC boundaries
boundary_plot = (
    boundary_gdf

    ### plot
    .hvplot(
        geo = True, 
        fill_color = None, 
        line_color = 'black',
        )
)


### combine plots
combined_plot = ndvi_yr_plot * boundary_plot

### wrap plots in Panel layout
pn.panel(combined_plot, widget_location = 'bottom')

BokehModel(combine_events=True, render_bundle={'docs_json': {'0acf743b-badb-4ea4-bb64-fea4955f128f': {'version…

### Reflect and Respond #6:
What do you observe? Don’t forget to write a headline and description of your plot!

## Wrap up

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [16]:
%store july_ndvi_df ndvi_in_da ndvi_out_da

Stored 'july_ndvi_df' (DataFrame)
Stored 'ndvi_in_da' (Dataset)
Stored 'ndvi_out_da' (Dataset)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!